# GPT PT-BR v8 — treino com livros + Q&A

Adicionados ao corpus:
- `data/qa_dolly_ptbr.txt` (~30k pares Pergunta/Resposta, Dolly via LibreTranslate)
- `data/qa_alpaca_ptbr.txt` (~15k pares, Alpaca via OPUS-MT)

Formato dos Q&A: cada exemplo termina em `<|endoftext|>` pro modelo aprender o fim de resposta.

**Pra começar do zero:** simplesmente rode o notebook (não existe `checkpoint_gpt_v8.pth` ainda).

**Pra continuar do V7:** antes de rodar, execute `cp checkpoint_gpt_v7.pth checkpoint_gpt_v8.pth` no terminal. O `load_checkpoint` vai retomar dali.

Depois de treinar, o webapp pega automaticamente o v8 (o backend prefere o maior v*).

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import tiktoken
import matplotlib.pyplot as plt
import time
import os
import glob

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Rodando no dispositivo: {device}")

In [ ]:
# HIPERPARÂMETROS v8 — arquitetura igual v7 (dá pra continuar do checkpoint v7)

block_size = 256
batch_size = 16

# Corpus cresceu ~6x (livros + 45k Q&A). Mais iters ajuda a ver mais variedade.
max_iters = 15000

learning_rate = 3e-4
eval_iters = 200

# Mesma arquitetura do v6/v7 — 52.93M params
n_embd = 384
n_layer = 8
n_head = 6
dropout = 0.1

In [ ]:
# Lê TODOS os .txt de data/ — livros + datasets Q&A adicionados em v8
files = sorted(glob.glob('data/*.txt'))
print(f"Encontrados {len(files)} arquivos para treinamento:")
for f in files:
    size_kb = os.path.getsize(f) / 1024
    print(f"  - {f} ({size_kb:.0f} KB)")

# Blacklist: cabeçalhos repetidos + marcadores do BrWaC/MWETOOLKIT que contaminaram o v7
blacklist = [
    "Página", "Page",
    "Colleen Hoover", "Machado de Assis",
    "Sumário", "Capítulo", "Copyright",
    "Todos os direitos reservados",
    "# MWETOOLKIT",
]

all_text_content = ""

for file_name in files:
    try:
        with open(file_name, 'r', encoding='utf-8') as f:
            raw_text = f.read()
            clean_lines = []
            for line in raw_text.split('\n'):
                line = line.strip()
                if not line:
                    continue
                if line.isdigit():
                    continue
                if any(bad in line for bad in blacklist):
                    continue
                if len(line) < 2 and line not in ['.', '?', '!', '—']:
                    continue
                clean_lines.append(line)
            
            book_text = "\n".join(clean_lines)
            # Os arquivos qa_*.txt já contêm `<|endoftext|>` entre pares.
            # Pra livros (sem esse marcador), adicionamos um no fim do arquivo.
            if "<|endoftext|>" not in book_text:
                book_text += " <|endoftext|>"
            
            all_text_content += book_text + "\n"
            print(f"  processado: {os.path.basename(file_name)} ({len(clean_lines)} linhas úteis)")
    except Exception as e:
        print(f"erro lendo {file_name}: {e}")

text = all_text_content
print(f"\nCorpus v8 final: {len(text)/1e6:.2f} MB de texto.")

In [ ]:
print("Tokenizando com GPT-2 BPE")
enc = tiktoken.get_encoding("gpt2")
encoded_data = enc.encode(text, allowed_special={"<|endoftext|>"})
vocab_size = enc.n_vocab
print(f"Vocab size: {vocab_size}")
print(f"Total tokens: {len(encoded_data):,}")

In [ ]:
data = torch.tensor(encoded_data, dtype=torch.long)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]
print(f"Train: {len(train_data):,} tokens | Val: {len(val_data):,} tokens")

In [ ]:
def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)

@torch.no_grad()
def estimate_metrics():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        accuracies = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
            probs = F.softmax(logits, dim=-1)
            pred = torch.argmax(probs, dim=-1)
            acc = (pred == Y.view(-1)).float().mean()
            accuracies[k] = acc.item()
        out[split] = {'loss': losses.mean(), 'acc': accuracies.mean()}
    model.train()
    return out

class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x); q = self.query(x)
        wei = q @ k.transpose(-2, -1) * C**-0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        v = self.value(x)
        return wei @ v

class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.dropout(self.proj(out))

class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )
    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd); self.ln2 = nn.LayerNorm(n_embd)
    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

class GPTLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)
        self.apply(self._init_weights)
    
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
    
    def forward(self, index, targets=None):
        B, T = index.shape
        tok_emb = self.token_embedding_table(index)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss
    
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None, stop_token_id=None):
        """Gera tokens. Se stop_token_id for passado, para ao emitir esse token."""
        for _ in range(max_new_tokens):
            idx_cond = idx if idx.size(1) <= block_size else idx[:, -block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
            if stop_token_id is not None and idx_next.item() == stop_token_id:
                break
        return idx

In [ ]:
CHECKPOINT_FILE = 'checkpoint_gpt_v8.pth'

def save_checkpoint(step, model, optimizer, loss_train, loss_val, acc_train, acc_val, filename=CHECKPOINT_FILE):
    checkpoint = {
        'step': step,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss_train': loss_train,
        'loss_val': loss_val,
        'acc_train': acc_train,
        'acc_val': acc_val,
    }
    torch.save(checkpoint, filename)
    print(f"  checkpoint salvo em step {step}")

def load_checkpoint(model, optimizer, filename=CHECKPOINT_FILE):
    if os.path.exists(filename):
        print(f"Carregando checkpoint '{filename}'...")
        ck = torch.load(filename, map_location=device, weights_only=False)
        model.load_state_dict(ck['model_state_dict'])
        optimizer.load_state_dict(ck['optimizer_state_dict'])
        step = ck['step']
        print(f"Retomando do step {step}")
        return step, ck.get('loss_train', []), ck.get('loss_val', []), ck.get('acc_train', []), ck.get('acc_val', [])
    print("Sem checkpoint — treinando do zero.")
    return 0, [], [], [], []

model = GPTLanguageModel()
m = model.to(device)
print(f"Modelo v8 criado com {sum(p.numel() for p in m.parameters())/1e6:.2f}M parâmetros")

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max_iters, eta_min=1e-5)

start_iter, loss_history_train, loss_history_val, acc_history_train, acc_history_val = load_checkpoint(model, optimizer)

print("Iniciando treino...")
start_time = time.time()

for iter in range(start_iter, max_iters):
    if iter % eval_iters == 0:
        metrics = estimate_metrics()
        t_loss, v_loss = metrics['train']['loss'], metrics['val']['loss']
        t_acc, v_acc = metrics['train']['acc'], metrics['val']['acc']
        print(f"step {iter}: loss {t_loss:.3f}/{v_loss:.3f} | acc {t_acc:.3f}/{v_acc:.3f}")
        loss_history_train.append(t_loss)
        loss_history_val.append(v_loss)
        acc_history_train.append(t_acc)
        acc_history_val.append(v_acc)
        save_checkpoint(iter, model, optimizer, loss_history_train, loss_history_val, acc_history_train, acc_history_val)
    
    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    scheduler.step()

elapsed = (time.time() - start_time) / 60
print(f"\nTreino finalizado em {elapsed:.2f} minutos.")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

ax1.plot(loss_history_train, label='Treino')
ax1.plot(loss_history_val, label='Validação')
ax1.set_title('Loss (Entropia Cruzada) — v8')
ax1.set_xlabel(f'Iterações (x {eval_iters})')
ax1.set_ylabel('Loss')
ax1.legend()
ax1.grid(True)

ax2.plot(acc_history_train, label='Treino')
ax2.plot(acc_history_val, label='Validação')
ax2.set_title('Accuracy de tokens — v8')
ax2.set_xlabel(f'Iterações (x {eval_iters})')
ax2.set_ylabel('Accuracy')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.savefig('outputs/grafico_metrics_v8.png')
print('salvo em outputs/grafico_metrics_v8.png')

In [ ]:
# TESTE DE GERAÇÃO — formato Q&A + continuação narrativa

print("=" * 40)
print("GERAÇÃO v8 — TESTE Q&A")
print("=" * 40)

model.eval()
eot_id = enc.eot_token  # 50256 — marca fim de resposta

def gerar(prompt, max_new=200, temperature=0.7, top_k=50):
    ids = enc.encode(prompt) if prompt else [0]
    ctx = torch.tensor(ids, dtype=torch.long, device=device).unsqueeze(0)
    out = model.generate(ctx, max_new_tokens=max_new, temperature=temperature, top_k=top_k, stop_token_id=eot_id)
    full = enc.decode(out[0].tolist())
    # Remove o eot final, se presente
    if full.endswith('<|endoftext|>'):
        full = full[:-len('<|endoftext|>')]
    return full

# Teste 1: formato Q&A
for prompt in [
    "Pergunta: O que é inteligência artificial?\n\nResposta:",
    "Pergunta: Quem foi Machado de Assis?\n\nResposta:",
    "Pergunta: Como fazer um bolo de chocolate?\n\nResposta:",
]:
    print("\n" + "-" * 40)
    print(gerar(prompt, max_new=150))

# Teste 2: continuação narrativa (estilo livros)
print("\n" + "=" * 40)
print("GERAÇÃO v8 — CONTINUAÇÃO")
print("=" * 40)
for prompt in [
    "Era uma vez um menino que",
    "Na pequena vila, o velho capitão",
]:
    print("\n" + "-" * 40)
    print(gerar(prompt, max_new=120))

# Salva state_dict limpo (sem optimizer) pra inferência
torch.save(model.state_dict(), 'gpt_ptbr_v8.pth')
print("\nModelo final salvo em gpt_ptbr_v8.pth")